# Homework 5 Solution

This notebook provides one example solution for the PCA assignment.


**Example choice:** This solution uses **Germany** for **2005-2020**.

The idea is to build a composite development index from indicators that reflect:
- economic development;
- social progress;
- environmental sustainability.

You can replace the country later if you want to build another valid solution.


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "homework_05"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR


## Task 1. Read the assignment template and WDI data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

assignment_df = pd.read_csv(DATA_DIR / "wdi" / "country_assignment_template.csv")
wdi_df = pd.read_csv(DATA_DIR / "wdi" / "WDI_course_subset.csv")

assignment_df.head()


In [ ]:
country_name = "Germany"
start_year = 2005
end_year = 2020

country_raw = wdi_df[wdi_df["Country Name"] == country_name]

panel_data = country_raw.drop(columns="Indicator Code").melt(
    id_vars=["Country Name", "Country Code", "Indicator Name"],
    var_name="Year",
).pivot_table(
    values="value",
    index=["Country Name", "Country Code", "Year"],
    columns="Indicator Name",
).reset_index().rename_axis("", axis=1)

panel_data["Year"] = panel_data["Year"].astype(str).astype(int)
panel_data = panel_data.query("@start_year <= Year <= @end_year").copy()

panel_data.head()


## Task 2. Select indicators based on economic logic

This solution uses six indicators:

1. **GDP per capita (constant 2015 US$)**: captures income and productive capacity.
2. **Life expectancy at birth**: captures health and social welfare.
3. **Government expenditure on education (% of GDP)**: captures public investment in human capital.
4. **Renewable energy consumption (% of total final energy consumption)**: captures energy transition and sustainability.
5. **PM2.5 air pollution**: captures environmental quality; lower pollution is better.
6. **Infant mortality rate**: captures basic public health; lower mortality is better.

For PCA, it is helpful if all variables move in the same welfare direction, so the two negative indicators will be reversed before standardization.


In [ ]:
indicator_info = pd.DataFrame(
    {
        "dimension": [
            "Economic development",
            "Social progress",
            "Social progress",
            "Environmental sustainability",
            "Environmental sustainability",
            "Social progress",
        ],
        "direction": ["positive", "positive", "positive", "positive", "negative", "negative"],
    },
    index=[
        "GDP per capita (constant 2015 US$)",
        "Life expectancy at birth, total (years)",
        "Government expenditure on education, total (% of GDP)",
        "Renewable energy consumption (% of total final energy consumption)",
        "PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)",
        "Mortality rate, infant (per 1,000 live births)",
    ],
)

selected_data = panel_data[["Country Name", "Country Code", "Year", *indicator_info.index]].copy()
selected_data[indicator_info.index] = (
    selected_data[indicator_info.index]
    .infer_objects(copy=False)
    .interpolate(method="linear", limit_direction="both")
)

negative_indicators = indicator_info.query("direction == 'negative'").index.tolist()
selected_data[negative_indicators] = -selected_data[negative_indicators]

selected_data.to_csv(MODULE_OUTPUT_DIR / "Homework5_selected_indicators.csv", index=False)
selected_data.head()


In [ ]:
missing_summary = selected_data[indicator_info.index].isna().sum().sort_values()
missing_summary


## Task 3. Standardize the variables and run PCA


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

X = selected_data[indicator_info.index].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmo_all, kmo_model = calculate_kmo(X_scaled)
chi_square_value, p_value = calculate_bartlett_sphericity(X_scaled)

pca = PCA()
component_scores = pca.fit_transform(X_scaled)

pca_summary = pd.DataFrame(
    {
        "Component": [f"PC{i+1}" for i in range(len(indicator_info))],
        "Eigenvalue": pca.explained_variance_,
        "Variance contribution rate": pca.explained_variance_ratio_,
        "Cumulative variance contribution rate": pca.explained_variance_ratio_.cumsum(),
    }
)

loadings = pd.DataFrame(
    pca.components_.T,
    index=indicator_info.index,
    columns=[f"PC{i+1}" for i in range(len(indicator_info))],
)

kmo_table = pd.DataFrame({
    "Statistic": ["KMO", "Bartlett chi-square", "Bartlett p-value"],
    "Value": [kmo_model, chi_square_value, p_value],
})

pca_summary.to_csv(MODULE_OUTPUT_DIR / "Homework5_pca_summary.csv", index=False)
loadings.to_csv(MODULE_OUTPUT_DIR / "Homework5_pca_loadings.csv")
kmo_table.to_csv(MODULE_OUTPUT_DIR / "Homework5_kmo_bartlett.csv", index=False)

kmo_table


In [ ]:
pca_summary


In [ ]:
loadings.iloc[:, :3].round(4)


### Retain the first two principal components

In this example, the cumulative variance contribution rate of the first two principal components is already above 85%, so keeping **PC1** and **PC2** is enough for a compact composite index.


In [ ]:
n_components = int(np.argmax(pca_summary["Cumulative variance contribution rate"] >= 0.85) + 1)
weights = pca.explained_variance_ratio_[:n_components]

development_index = component_scores[:, :n_components] @ weights

# PCA signs are arbitrary, so orient the index so that it is positively related to GDP per capita.
if np.corrcoef(development_index, selected_data["GDP per capita (constant 2015 US$)"])[0, 1] < 0:
    development_index = -development_index
    component_scores[:, :n_components] = -component_scores[:, :n_components]
    loadings.iloc[:, :n_components] = -loadings.iloc[:, :n_components]

result_df = selected_data[["Country Name", "Country Code", "Year"]].copy()
for i in range(n_components):
    result_df[f"PC{i+1}"] = component_scores[:, i]
result_df["development_index"] = development_index
result_df["development_rank"] = result_df["development_index"].rank(ascending=False, method="dense")

result_df.to_csv(MODULE_OUTPUT_DIR / "Homework5_development_index.csv", index=False)
result_df


## Task 4. Export figures for the written report

A written report can use these figures to explain the PCA process and the time trend of the composite development index.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(pca_summary) + 1), pca_summary["Eigenvalue"], marker="o")
plt.axhline(y=1, color="red", linestyle="--", linewidth=1)
plt.xlabel("Principal component")
plt.ylabel("Eigenvalue")
plt.title(f"{country_name}: Scree Plot")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "Homework5_scree_plot.png", bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.lineplot(data=result_df, x="Year", y="development_index", marker="o")
plt.axhline(y=0, color="gray", linestyle="--", linewidth=1)
plt.title(f"{country_name}: PCA-Based Development Index")
plt.ylabel("Composite index")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "Homework5_development_index_line.png", bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(loadings.iloc[:, :n_components], annot=True, cmap="RdBu_r", center=0)
plt.title(f"{country_name}: PCA Loadings")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "Homework5_pca_loadings_heatmap.png", bbox_inches="tight")
plt.show()


## Interpretation notes for the report

One reasonable interpretation is:

- **PC1** behaves like a broad development factor: it loads positively on income, life expectancy, renewable energy, better air quality, and lower infant mortality.
- **PC2** mainly separates periods with relatively stronger public education spending from periods where other indicators improved more quickly.
- The final composite index rises over time, which is consistent with Germany's steady gains in income, health, cleaner air, and energy transition during 2005-2020.

For the report, you can also compare Germany with another country by discussing differences in raw indicators, even if the PCA itself is estimated for Germany only.


In [ ]:
report_materials = pd.DataFrame(
    {
        "file": [
            "Homework5_selected_indicators.csv",
            "Homework5_kmo_bartlett.csv",
            "Homework5_pca_summary.csv",
            "Homework5_pca_loadings.csv",
            "Homework5_development_index.csv",
            "Homework5_scree_plot.png",
            "Homework5_development_index_line.png",
            "Homework5_pca_loadings_heatmap.png",
        ]
    }
)

report_materials
